# 12 – linker.py results
Reads the output of `step4_run_linker.py` (folder `results/`) and answers:
1. How many merged AI PRs got a candidate fix (14 / 30 days)? Per agent.
2. Who wrote the fix: an AI agent or a human?
3. How fast does the fix come?
4. Do we find the same pairs as the paper (2,510 candidates, 334 confirmed by the judge)?
5. Do we find our known fixes (323 clean pairs, 948 AI PRs fixed later)?
6. A sample of pairs for the manual analysis.

In [ ]:
import os, json, sys
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, "../..")
from common import bar, show_numbers, save_table, save_chart
from paths import REPO, RESULTS
D = os.path.join("..", "..", "..", "data")
per_pr = pd.read_csv(os.path.join(RESULTS, "per_pr.csv"))
per_agent = pd.read_csv(os.path.join(RESULTS, "per_agent.csv"))
pairs = pd.read_csv(os.path.join(RESULTS, "pairs.csv"))
print(f"merged AI PRs: {len(per_pr):,} | candidate pairs: {len(pairs):,}")

## 1. AI PRs with a candidate fix, per agent

In [ ]:
t = per_agent[["agent", "n_prs", "pct_followup_14d", "pct_followup_30d", "pct_followup_30d_unc"]].round(1)
t.columns = ["agent", "merged AI PRs", "% fix within 14 days", "% fix within 30 days", "% 30 days (uncensored)"]
display(save_table(t, "1_per_agent"))
fixed = per_pr[per_pr.has_followup_30d == 1].agent.value_counts()
bar(fixed, "AI PRs with a candidate fix within 30 days", "1_fixed_per_agent.png", "AI PRs")

## 2. Who wrote the fix?

In [ ]:
who = pd.crosstab(pairs.ai_agent, pairs.fixed_by, margins=True, margins_name="TOTAL")
display(save_table(who.reset_index(), "2_who_fixes"))
bar(pairs.fixed_by.value_counts(), "Who wrote the fix PR", "2_who_fixes.png", "pairs", pct=True)

## 3. How fast does the fix come?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(pairs.days_between, bins=30, color="#1F4E79")
ax.set_xlabel("days between AI PR merge and fix PR merge"); ax.set_ylabel("pairs")
ax.set_title("Time to fix")
save_chart(fig, "3_days_to_fix.png")
display(pairs.days_between.describe().round(1).to_frame("days"))

## 4. Same pairs as the paper?\nPaper: 2,510 candidate pairs; the LLM judge marked 334 as *Direct fix*.

In [ ]:
J = os.path.join(REPO, "data", "judge")
paper = pd.DataFrame(json.load(open(os.path.join(J, "rq1_judge_cases_population_12k.json"), encoding="utf-8")))
verd = pd.read_csv(os.path.join(J, "rq1_llm_judge_pop_anthropic_claude-opus-4-8_low.csv"))
direct = verd[verd.label.str.startswith("Direct")]
key = lambda p, q: p.astype(str) + " -> " + q.astype(str)
ours = set(key(pairs.ai_pr_url, pairs.fix_pr_url))
paper_k = set(key(paper.p_url, paper.q_url))
direct_k = set(key(direct.p_url, direct.q_url))
comp = pd.DataFrame([
    ["Candidate pairs", len(paper_k), len(ours), len(ours & paper_k)],
    ["Direct fixes (judge)", len(direct_k), "", len(ours & direct_k)],
], columns=["", "paper", "our run", "found in both"])
display(save_table(comp, "4_compare_paper"))
print(f"We find {len(ours & paper_k) / max(len(paper_k), 1):.1%} of the paper's candidates "
      f"and {len(ours & direct_k) / max(len(direct_k), 1):.1%} of its direct fixes.")

## 5. Do we find our known fixes?

In [ ]:
clean = pd.read_csv(os.path.join(D, "CLEAN_PR_DATASET", "CLEAN_AI_PR_FIXES.csv"))
clean_k = set(key(clean.ai_pr_url, clean.fix_pr_url))
ai = pd.read_csv(os.path.join(D, "AI_PRS_DATASET", "AI_PRS_DATASET.csv"), low_memory=False)
known = ai[ai.fixed_later == True]
found_ai = set(pairs.ai_pr_url)
k = pd.DataFrame([
    ["323 clean AI PR -> fix pairs (Who Finishes)", len(clean_k), len(ours & clean_k)],
    ["AI PRs fixed later (Who Finishes + Khemissi)", len(known), int(known.pr_url.isin(found_ai).sum())],
], columns=["known fixes", "how many", "found by linker.py"])
display(save_table(k, "5_known_fixes"))

## 6. Sample for the manual analysis\nRandom 50 pairs (fixed seed), Python first if available.

In [ ]:
langs = ai.set_index("pr_url")["language"] if "language" in ai else pd.Series(dtype=str)
pairs["language"] = pairs.ai_pr_url.map(langs)
pool = pairs[pairs.language == "Python"] if (pairs.language == "Python").sum() >= 50 else pairs
sample = pool.sample(min(50, len(pool)), random_state=42)
for c in ["why_fixed (Q1)", "what_was_wrong (Q2)", "how_fixed (Q3)", "notes"]:
    sample[c] = ""
display(sample.head(10))
with pd.ExcelWriter(os.path.join("tables", "linker_results.xlsx")) as xw:
    t.to_excel(xw, sheet_name="1 Per agent", index=False)
    who.to_excel(xw, sheet_name="2 Who fixes")
    comp.to_excel(xw, sheet_name="4 vs paper", index=False)
    k.to_excel(xw, sheet_name="5 Known fixes", index=False)
    pairs.to_excel(xw, sheet_name="All pairs", index=False)
    sample.to_excel(xw, sheet_name="Sample to annotate", index=False)
print("Saved tables/linker_results.xlsx")